### scrapy
+ 비동기 방식으로 데이터 수집 -> 데이터 수집 속도가 빠름

In [1]:
import scrapy, requests
from scrapy.http import TextResponse

In [2]:
# Gmarket 베스트 200 상품 데이터 수집

#### 1.scrapy 프로젝트 생성

In [3]:
!scrapy startproject gmarket 
#gmarket : 프로젝트 이름

Error: scrapy.cfg already exists in C:\Users\User\Desktop\김시연\실습\웹크롤링\day3\code\gmarket


In [4]:
!tree gmarket /f

폴더 PATH의 목록입니다.
볼륨 일련 번호가 000000EC 7C6C:FC35입니다.
C:\USERS\USER\DESKTOP\김시연\실습\웹크롤링\DAY3\CODE\GMARKET
│  items.py
│  scrapy.cfg
│  
└─gmarket
    │  items.py
    │  middlewares.py
    │  pipelines.py
    │  settings.py
    │  __init__.py
    │  
    └─spiders
            __init__.py
            


In [5]:
# items.py : 수집할 데이터의 구조 정의
# middlewares.py : 데이터를 수집할때 headers 정보와 같은 내용을 설정
# pipelines.py: 데이터를 수집한 후에 코드 실행 정의
# settings.py : 크롤링 설정 -> 크롤링 시간 텀, robots.txt 규칙
# spiders: 디렉토리, 크롤링 절차 정의

#### 2.xpath 찾기 : 링크, 상세 페이지의 데이터

In [6]:
#링크 데이터 수집 

In [7]:
request=requests.get("http://corners.gmarket.co.kr/Bestsellers")
response=TextResponse(request.url,body=request.text,encoding="utf-8")

In [12]:
links = response.xpath('//*[@id="gBestWrap"]/div/div[3]/div/ul/li/a/@href').extract()


In [13]:
links[:2]

['http://item.gmarket.co.kr/Item?goodscode=2519604129&ver=637953132312445146',
 'http://item.gmarket.co.kr/Item?goodscode=2520466046&ver=637953132312445146']

In [10]:
#상세 페이지 데이터 수집

In [14]:
link = links[0]
request = requests.get(link)
response = TextResponse(request.url, body=request.text, encoding="utf-8")

In [15]:
title=response.xpath('//*[@id="itemcase_basic"]/div/h1/text()')[0].extract()
price=response.xpath('//*[@id="itemcase_basic"]/div/p/span/strong/text()')[0].extract()
title,price

('브라운 면도기 세정액 CCR5+1 ', '25,030')

#### 3.items.py 코드 작성, model(데이터의 구조 : 수집할 데이터의 컬럼을 정의)

In [25]:
%%writefile gmarket/gmarket/items.py
import scrapy

class GmarketItem(scrapy.Item):
    title = scrapy.Field()
    price = scrapy.Field()
    link = scrapy.Field()

Overwriting gmarket/gmarket/items.py


#### 4.spider.py : 코드 작성, 크롤링 정차 정의

In [26]:
%%writefile gmarket/gmarket/spiders/spider.py
import scrapy
from gmarket.items import GmarketItem

class GMSpider(scrapy.Spider):
    name = "GMB"
    allow_domain = ["gmarket.co.kr"]
    start_urls = ["http://corners.gmarket.co.kr/Bestsellers"]
    
    def parse(self, response):
        links = response.xpath('//*[@id="gBestWrap"]/div/div[3]/div/ul/li/a/@href').extract()
        for link in links[:20]:
            yield scrapy.Request(link, callback=self.parse_content)
    
    def parse_content(self, response):
        item = GmarketItem()
        item["title"] = response.xpath('//*[@id="itemcase_basic"]/div[1]/h1/text()')[0].extract()
        item["price"] = response.xpath('//*[@id="itemcase_basic"]/div[1]/p/span/strong/text()')[0].extract()
        item["link"] = response.url
        yield item

Overwriting gmarket/gmarket/spiders/spider.py


#### 5.scrapy 프로젝트 실행

In [18]:
#gmarket 디렉토리
#scrapy crawl GMB -o items.csv -> 터미널에서 실행

In [23]:
%pwd

'C:\\Users\\User\\Desktop\\김시연\\실습\\웹크롤링\\day3\\code'

In [27]:
pd.read_csv("gmarket/items.csv").tail(2)

,link,price,title
18,http://item.gmarket.co.kr/Item?goodscode=23678...,"17,900",군산 반건조 서대 박대구이 특대 5미 2개(쿠폰15940)
19,http://item.gmarket.co.kr/Item?goodscode=21414...,"11,500",붕어빵 아기물티슈 썸머에디션 핫한 고평량 70매 10팩 70gsm
